#### Session 3: Multi-model forecasting group

This notebook compares a diverse set of forecasting models, from simple baselines to classical statistical methods and Prophet.

The goal is not only to measure accuracy, but also to create meaningful variation in error patterns for interpretation, bias analysis, and explainability.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
from pathlib import Path

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from prophet import Prophet
    HAS_PROPHET = True
except ImportError:
    HAS_PROPHET = False
    print("Prophet not installed — skipping Prophet model")

from sklearn.linear_model import LinearRegression


## Load data

The series is loaded as a univariate time series with a `DatetimeIndex`.  
For consistency across all models, the data is resampled to daily frequency.

In [2]:
DATA_DIR = Path("../src/data")
OUTPUT_DIR = Path("../src/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "energy"


In [3]:
def load_series() -> pd.Series:
    for csv_path in [
        DATA_DIR / "cleaned_aep_hourly.csv",
        DATA_DIR / "raw" / "AEP_hourly.csv",
    ]:
        if csv_path.exists():
            df = pd.read_csv(csv_path, parse_dates=["Datetime"], index_col="Datetime")
            s = df.squeeze().sort_index()
            print(f"Loaded {len(s)} hourly records from {csv_path.name}")
            break
    else:
        print("Real data not found — using synthetic fallback")
        rng = np.random.default_rng(42)
        idx = pd.date_range("2015-01-01", periods=8760, freq="h")
        trend = np.linspace(12000, 14000, 8760)
        seasonal_daily  = 2000 * np.sin(2 * np.pi * np.arange(8760) / 24)
        seasonal_weekly = 800  * np.sin(2 * np.pi * np.arange(8760) / (24 * 7))
        noise = rng.normal(0, 300, 8760)
        s = pd.Series(trend + seasonal_daily + seasonal_weekly + noise,
                      index=idx, name="AEP_MW")

    s = s.resample("D").mean().dropna()
    return s

series = load_series()
print(f"Daily series: {len(series)} days | {series.index[0].date()} → {series.index[-1].date()}")

Loaded 121296 hourly records from cleaned_aep_hourly.csv
Daily series: 5055 days | 2004-10-01 → 2018-08-03


## Rolling evaluation

The last 90 days of data are used for evaluation, split into 3 consecutive 30-day folds.
Each fold trains on all data up to that point (expanding window) and tests on the next 30 days.
Metrics are averaged across folds to produce more robust estimates than a single split.

In [4]:
def compute_metrics(actual, predicted, model_name):
    actual    = np.asarray(actual,    dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    safe      = np.where(actual == 0, np.nan, actual)

    mae  = float(np.nanmean(np.abs(actual - predicted)))
    rmse = float(np.sqrt(np.nanmean((actual - predicted) ** 2)))
    mape = float(np.nanmean(np.abs((actual - predicted) / safe)) * 100)
    mpe  = float(np.nanmean((actual - predicted) / safe) * 100)
    da   = float(np.mean(np.sign(np.diff(actual)) == np.sign(np.diff(predicted))) * 100) if len(actual) > 1 else float("nan")

    return {"model": model_name,
            "MAE": round(mae, 2), "RMSE": round(rmse, 2),
            "MAPE": round(mape, 4), "MPE": round(mpe, 4), "DA": round(da, 2)}

def store_forecast(name, forecast, test_series):
    return [{"date": str(idx.date()), "actual": round(float(act), 2),
             "predicted": round(float(forecast[i]), 2), "model": name}
            for i, (idx, act) in enumerate(test_series.items())]

In [5]:
N_FOLDS   = 3
FOLD_SIZE = 30
SEASON    = 7

series = load_series()
total_test = N_FOLDS * FOLD_SIZE
print(f"Series: {len(series)} days | Rolling eval: {N_FOLDS} folds × {FOLD_SIZE} days (last {total_test} days evaluated)")

Loaded 121296 hourly records from cleaned_aep_hourly.csv
Series: 5055 days | Rolling eval: 3 folds × 30 days (last 90 days evaluated)


## Model 1: Naive baseline

This forecast repeats the last observed value and serves as the simplest benchmark.


In [6]:
def run_all_models(train, test):
    """Train all 7 models on train, evaluate on test. Returns (metrics_list, forecasts_list)."""
    horizon = len(test)
    metrics, forecasts = [], []

    # Naive
    naive_pred = np.full(horizon, float(train.iloc[-1]))
    metrics.append(compute_metrics(test.values, naive_pred, "Naive"))
    forecasts.extend(store_forecast("Naive", naive_pred, test))
    print("  ✓ Naive")

    # Seasonal Naive
    snaive_pred = np.array([train.iloc[-SEASON + (i % SEASON)] for i in range(horizon)])
    metrics.append(compute_metrics(test.values, snaive_pred, "Seasonal Naive"))
    forecasts.extend(store_forecast("Seasonal Naive", snaive_pred, test))
    print("  ✓ Seasonal Naive")

    # Linear Regression
    X_tr = np.arange(len(train)).reshape(-1, 1)
    X_te = np.arange(len(train), len(train) + horizon).reshape(-1, 1)
    lr_pred = LinearRegression().fit(X_tr, train.values).predict(X_te)
    metrics.append(compute_metrics(test.values, lr_pred, "Linear Regression"))
    forecasts.extend(store_forecast("Linear Regression", lr_pred, test))
    print("  ✓ Linear Regression")

    # ETS
    ets_pred = ExponentialSmoothing(
        train, trend="add", seasonal="add", seasonal_periods=SEASON,
        initialization_method="estimated"
    ).fit(optimized=True).forecast(horizon).values
    metrics.append(compute_metrics(test.values, ets_pred, "ETS"))
    forecasts.extend(store_forecast("ETS", ets_pred, test))
    print("  ✓ ETS")

    # HWES (damped)
    hwes_pred = ExponentialSmoothing(
        train, trend="add", damped_trend=True, seasonal="add", seasonal_periods=SEASON,
        initialization_method="estimated"
    ).fit(optimized=True).forecast(horizon).values
    metrics.append(compute_metrics(test.values, hwes_pred, "HWES (damped)"))
    forecasts.extend(store_forecast("HWES (damped)", hwes_pred, test))
    print("  ✓ HWES (damped)")

    # SARIMA
    try:
        sarima_data = train.iloc[-min(730, len(train)):]
        sarima_pred = SARIMAX(
            sarima_data, order=(1,1,1), seasonal_order=(1,1,1,SEASON),
            enforce_stationarity=False, enforce_invertibility=False
        ).fit(disp=False).forecast(steps=horizon).values
        metrics.append(compute_metrics(test.values, sarima_pred, "SARIMA"))
        forecasts.extend(store_forecast("SARIMA", sarima_pred, test))
        print("  ✓ SARIMA")
    except Exception as e:
        print(f"  ✗ SARIMA failed: {e}")

    # Prophet
    if HAS_PROPHET:
        try:
            pm = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, interval_width=0.95)
            pm.fit(pd.DataFrame({"ds": train.index, "y": train.values}))
            prophet_pred = pm.predict(pm.make_future_dataframe(periods=horizon, freq="D"))["yhat"].iloc[-horizon:].values
            metrics.append(compute_metrics(test.values, prophet_pred, "Prophet"))
            forecasts.extend(store_forecast("Prophet", prophet_pred, test))
            print("  ✓ Prophet")
        except Exception as e:
            print(f"  ✗ Prophet failed: {e}")

    return metrics, forecasts

## Model 2: Seasonal Naive baseline

This model reuses the value from the same point in the previous weekly cycle and captures recurring seasonality.


In [7]:
all_fold_metrics   = []   # list of (fold_idx, metrics_dict)
last_fold_forecasts = []

for fold in range(N_FOLDS):
    test_end   = len(series) - fold * FOLD_SIZE
    test_start = test_end - FOLD_SIZE
    train_fold = series.iloc[:test_start]
    test_fold  = series.iloc[test_start:test_end]

    print(f"\n─── Fold {fold+1}/{N_FOLDS} | train: {len(train_fold)} days | "
          f"test: {test_fold.index[0].date()} → {test_fold.index[-1].date()} ───")

    fold_m, fold_f = run_all_models(train_fold, test_fold)
    all_fold_metrics.extend((fold, m) for m in fold_m)
    if fold == 0:                     # most recent fold → sparklines
        last_fold_forecasts = fold_f

print("\n✓ All folds complete")


─── Fold 1/3 | train: 5025 days | test: 2018-07-05 → 2018-08-03 ───
  ✓ Naive
  ✓ Seasonal Naive
  ✓ Linear Regression


  ✓ ETS


  ✓ HWES (damped)


14:23:15 - cmdstanpy - INFO - Chain [1] start processing


  ✓ SARIMA


...

  ✓ HWES (damped)


14:23:19 - cmdstanpy - INFO - Chain [1] start processing


  ✓ SARIMA


14:23:20 - cmdstanpy - INFO - Chain [1] done processing


  ✓ Prophet

✓ All folds complete


## Model 3: Linear Regression trend baseline

This model captures only a linear trend, without explicit seasonality.


In [8]:
# Average each metric across folds per model
model_names = [m["model"] for _, m in all_fold_metrics if _ == 0]
metrics_rows = []
for name in model_names:
    folds = [m for _, m in all_fold_metrics if m["model"] == name]
    metrics_rows.append({
        "model": name,
        "MAE":  round(np.mean([m["MAE"]  for m in folds]), 2),
        "RMSE": round(np.mean([m["RMSE"] for m in folds]), 2),
        "MAPE": round(np.mean([m["MAPE"] for m in folds]), 4),
        "MPE":  round(np.mean([m["MPE"]  for m in folds]), 4),
        "DA":   round(np.mean([m["DA"]   for m in folds]), 2),
    })

metrics_df   = pd.DataFrame(metrics_rows)
all_forecasts = last_fold_forecasts

print("Averaged metrics across", N_FOLDS, "folds:")
print(metrics_df.to_string(index=False))

Averaged metrics across 3 folds:
            model     MAE    RMSE    MAPE     MPE    DA
            Naive 1827.63 2165.77 11.9803  5.4190  0.00
   Seasonal Naive 1352.16 1719.28  8.9438 -0.5710 59.77
Linear Regression 1238.04 1505.50  7.9361  2.9084 50.58
              ETS 1546.33 1812.99 10.0185  5.2486 66.67
    HWES (damped) 1517.14 1779.22  9.8208  5.2220 67.82
           SARIMA 1435.00 1704.85  9.2087  4.4870 66.67
          Prophet 1091.53 1320.09  7.0529  2.6136 65.52


## Model 4: ETS

ETS combines error, trend, and seasonality through exponential smoothing.


## Model 5: Holt-Winters Exponential Smoothing

This version uses a damped trend so the forecast growth tapers off over time.


## Model 6: SARIMA

SARIMA extends ARIMA with seasonal structure and is often useful when repeated cycles matter.


## Model 7: Prophet

Prophet is included as a flexible decomposable model that can capture trend and seasonality patterns in a different way from the classical methods.


## Save outputs

The metrics and forecasts are saved for later interpretation and for any downstream visualization layer.


In [9]:
metrics_path   = OUTPUT_DIR / "metrics_all_models.json"
forecasts_path = OUTPUT_DIR / "forecasts_all_models.json"

metrics_df.to_json(metrics_path, orient="records", indent=2)
metrics_df.to_csv(OUTPUT_DIR / "metrics_all_models.csv", index=False)

with open(forecasts_path, "w") as f:
    json.dump(all_forecasts, f, indent=2)

print(metrics_df.to_string(index=False))
print(f"\nSaved metrics   → {metrics_path}")
print(f"Saved forecasts → {forecasts_path}")

            model     MAE    RMSE    MAPE     MPE    DA
            Naive 1827.63 2165.77 11.9803  5.4190  0.00
   Seasonal Naive 1352.16 1719.28  8.9438 -0.5710 59.77
Linear Regression 1238.04 1505.50  7.9361  2.9084 50.58
              ETS 1546.33 1812.99 10.0185  5.2486 66.67
    HWES (damped) 1517.14 1779.22  9.8208  5.2220 67.82
           SARIMA 1435.00 1704.85  9.2087  4.4870 66.67
          Prophet 1091.53 1320.09  7.0529  2.6136 65.52

Saved metrics   → ../src/data/metrics_all_models.json
Saved forecasts → ../src/data/forecasts_all_models.json


## Ranking and bias interpretation

The table below ranks models by MAE and summarizes forecast bias using MPE.


In [10]:
print("\n=== Ranking by MAE (lower is better) ===")
ranked = metrics_df.sort_values("MAE").reset_index(drop=True)
ranked.index += 1
print(ranked[["model", "MAE", "RMSE", "MAPE", "MPE", "DA"]].to_string())

print("\n=== Bias check (MPE: + = over-forecast, − = under-forecast) ===")
for _, row in metrics_df.iterrows():
    bias_label = "over-forecast" if row["MPE"] > 1 else ("under-forecast" if row["MPE"] < -1 else "unbiased")
    print(f"  {row['model']:25s}  MPE={row['MPE']:+.2f}%  → {bias_label}")



=== Ranking by MAE (lower is better) ===
               model      MAE     RMSE     MAPE     MPE     DA
1            Prophet  1091.53  1320.09   7.0529  2.6136  65.52
2  Linear Regression  1238.04  1505.50   7.9361  2.9084  50.58
3     Seasonal Naive  1352.16  1719.28   8.9438 -0.5710  59.77
4             SARIMA  1435.00  1704.85   9.2087  4.4870  66.67
5      HWES (damped)  1517.14  1779.22   9.8208  5.2220  67.82
6                ETS  1546.33  1812.99  10.0185  5.2486  66.67
7              Naive  1827.63  2165.77  11.9803  5.4190   0.00

=== Bias check (MPE: + = over-forecast, − = under-forecast) ===
  Naive                      MPE=+5.42%  → over-forecast
  Seasonal Naive             MPE=-0.57%  → unbiased
  Linear Regression          MPE=+2.91%  → over-forecast
  ETS                        MPE=+5.25%  → over-forecast
  HWES (damped)              MPE=+5.22%  → over-forecast
  SARIMA                     MPE=+4.49%  → over-forecast
  Prophet                    MPE=+2.61%  → over-fore

## Interpretation

The main purpose of this notebook is to compare model behavior across a diverse forecasting group.

The next step is to interpret not only which model performs best, but also which models are biased, too smooth, or too reactive.


#### - Naive is the simplest reference point and usually performs worst unless the series is very stable.
- Seasonal Naive often improves on Naive when weekly repetition is strong.
- Linear Regression captures trend but misses local seasonality.
- ETS and HWES are useful for smooth trend-seasonality structure, with HWES usually being more conservative because of damped trend.
- SARIMA can model richer autocorrelation and seasonality, but may be slower and more sensitive to parameter choice.
- Prophet is useful when the series has structured trend and seasonal components and often provides a strong comparison point for interpretation.
